# (부록) 로컬 소형 vs 로컬 대형 vs 클라우드 — Function Calling 신뢰도 비교

> 본 노트북은 `M02_3_mcp_a2a.ipynb` **섹션 7** 의 보충 자료입니다.
> *"도구 호출 실패는 **로컬이라서**가 아니라 **런타임(서버)** 과 **모델 크기** 때문"* 임을,
> **반복 실험**으로 **정량 비교**합니다.

## 비교 대상 — function calling 신뢰도의 **두 축**
| # | 런타임 · 모델 | 기대 |
|---|---|---|
| ①-A | **Ollama** · `qwen2.5:0.5b` (소형) | 유효 `tool_calls` ✅ (Ollama 가 도구 템플릿 적용) |
| ①-B | **llama.cpp** · `qwen2.5:0.5b` (**같은** 소형!) | malformed → 실패 ❌ (기본 서버는 도구 파싱 안 함) |
| ② | **Ollama** · `qwen3:8b` (대형) ★ | 안정 ✅ |
| ③ | **클라우드** · `google` (Gemini, 키 있으면 자동) | 안정 ✅ |

## 핵심 포인트 — 두 축으로 읽기
- **런타임/서버 축** → ①-A vs ①-B (**같은 모델, 다른 서버**):
  Ollama 는 OpenAI 호환 `/v1` 에서 소형 모델도 **네이티브 `tool_calls`** 로 만들지만,
  보조 서버 `llama-cpp-python`(기본)은 도구 호출을 구조화 파싱하지 않아 소형 모델이
  `<tool_call>{{ ... }}` 같은 **깨진 JSON** 을 흘려 실행 불가 ❌.
- **모델 능력 축** → ①-A vs ② (**같은 서버, 다른 모델**):
  큰 모델(`qwen3:8b`)일수록 다중 도구 태스크의 **완성도·인자 정확도**가 높음.

즉 *"로컬이라 안 된다"* 가 아니라, **적절한 런타임(Ollama) + 충분한 모델(`qwen3:8b`)** 이면
**로컬로도 안정적인 function calling 이 가능**합니다.

> **실행 환경**: Windows + CMD + Ollama(기본) + llama.cpp(보조, 실험 ①-B 에서만). 로컬 모델은 미리
> `ollama pull qwen2.5:0.5b` / `ollama pull qwen3:8b`. ①-B 의 GGUF 는 최초 1회 자동 다운로드(캐시).
> 상세는 `env_guides/M02_2_function_calling.md` 참고.

## 0. 환경 설정

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath(''))
import utils
utils.reload_env()

from agentic_lib import bootstrap
from agentic_lib.bootstrap import to_text          # 공급자 무관 응답 정규화(<think> 제거)

# 로컬(Ollama, OpenAI 호환) + 클라우드 비교용 패키지
utils.uv_install(['langchain', 'langchain-openai', 'langchain-google-genai',
                  'langchain-anthropic', 'openai'])
print('준비 완료')

LLM 공급자: nvidia
  NVIDIA build Key: 설정됨  /  Model: meta/llama-3.1-8b-instruct


[uv] 설치 완료: ['langchain', 'langchain-openai', 'langchain-google-genai', 'langchain-anthropic', 'openai']
준비 완료


## 1. 두 로컬 모델(Ollama) 준비 & 모델 팩토리

Ollama 는 OpenAI 호환 `/v1` 을 제공하므로, 로컬 모델을 바꾸는 것은 `ChatOpenAI(model=...)` 의
`model` 문자열만 바꾸는 일입니다. GGUF 다운로드나 별도 서버 기동이 **필요 없습니다.**

> 최초 1회 준비(CMD): `ollama pull qwen2.5:0.5b` (~0.4GB), `ollama pull qwen3:8b` (~5GB).

In [2]:
from langchain_openai import ChatOpenAI

# 두 로컬 모델 태그(모두 `ollama pull` 로 준비되어 있어야 함)
MODELS = {
    "small": {"label": "Ollama qwen2.5:0.5b", "model": "qwen2.5:0.5b"},
    "large": {"label": "Ollama qwen3:8b",     "model": "qwen3:8b"},
}

def make_llm(model, temperature=0.7):
    '''지정한 Ollama 모델에 도구를 바인딩한 LangChain LLM 을 만든다.

    세 실험 모두 동일한 OpenAI 호환 인터페이스를 쓰며, 로컬 모델 전환은 `model` 한 줄뿐이다.
    (TOOLS 는 다음 셀에서 정의되며, 호출 시점에 참조된다.)
    '''
    return ChatOpenAI(model=model, base_url=utils.OLLAMA_BASE_URL,
                      api_key="ollama", temperature=temperature).bind_tools(TOOLS)

## 2. 도구 정의 + 분류기(네이티브 `tool_calls` 우선, `<tool_call>` 파싱 폴백)

도구 호출이 **반드시 필요한** 요청을 준비하고, 응답을 다음 4가지로 분류합니다.

| 분류 | 의미 |
|------|------|
| `TOOL_OK` | **실행 가능한 도구 호출** 획득 — 네이티브 `tool_calls` 또는 파싱된 `<tool_call>` JSON ✅ |
| `MALFORMED` | 도구 호출 텍스트는 있으나 JSON 이 **깨져** 파싱 불가 ⚠️ |
| `NO_TOOL` | 도구를 시도하지 않고 그냥 말로 답함 |
| `ERROR` | 예외/500 |

`extract_calls()` 가 응답에서 실행 가능한 호출을 뽑아냅니다. Ollama + `qwen3:8b` 는 보통
**네이티브 `tool_calls`(source=native)** 로 오고, 소형 모델이 파싱 못한 채 본문에 흘리면
`<tool_call>` 정규식 파싱(source=parsed)으로 폴백합니다(소형 모델의 malformed 출력 관찰용).
반복 변동을 보기 위해 `temperature=0.7` 을 사용합니다.

In [3]:
import math, json as _json, re
from collections import Counter
from langchain.tools import tool
from langchain_core.messages import HumanMessage, SystemMessage

@tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 조회합니다."""
    return f"{city} 날씨: 맑음, 22도"

# eval 에 안전한 수학 심볼만 노출 (모델이 sqrt/pow 등을 그대로 쓰는 경우 대비)
_MATH = {k: getattr(math, k) for k in
         ['sqrt','pow','sin','cos','tan','log','log10','exp','pi','e','factorial','floor','ceil','fabs']}

@tool
def calculator(expression: str) -> str:
    """수학 계산을 수행합니다. (예: 'sqrt(144)', '12 * 12')"""
    try:
        return f"{expression} = {eval(expression, {'__builtins__': {}}, _MATH)}"
    except Exception as e:
        return f"오류: {e}"

TOOLS = [get_weather, calculator]
TOOL_FUNCS = {t.name: t for t in TOOLS}
# 이 요청은 '날씨 조회' + '제곱근 계산' 두 도구를 모두 필요로 한다(완성도 측정용).
REQUIRED_TOOLS = {"get_weather", "calculator"}
# qwen3 는 기본 thinking 모드 → /no_think 로 비활성화(도구 호출 출력이 깔끔해짐, 다른 모델엔 영향 없음)
PROMPT = [SystemMessage(content="반드시 제공된 도구를 사용해 답하세요. /no_think"),
          HumanMessage(content="서울 날씨를 알려주고 144의 제곱근을 계산해줘.")]

_TOOL_RE = re.compile(r"<tool_call>\s*(\{.*?\})\s*</tool_call>", re.DOTALL)

def parse_text_calls(text):
    """본문 텍스트에서 <tool_call> JSON 블록을 추출·검증해 (name, args) 리스트로 반환."""
    calls = []
    for m in _TOOL_RE.finditer(text):
        try:
            obj = _json.loads(m.group(1))
        except Exception:
            continue                                  # JSON 깨짐 → 실행 불가
        if isinstance(obj, dict) and "name" in obj:
            calls.append((obj["name"], obj.get("arguments", obj.get("parameters", {})) or {}))
    return calls

def extract_calls(resp):
    """실행 가능한 도구 호출을 뽑는다. 반환: (calls, source) — source: native|parsed|None"""
    native = getattr(resp, "tool_calls", None) or []
    if native:
        return [(t["name"], t.get("args", {})) for t in native], "native"
    parsed = parse_text_calls(to_text(resp.content))  # Ollama 가 파싱 못한 경우(원시 서버) 폴백
    if parsed:
        return parsed, "parsed"
    return [], None

def classify(resp):
    """응답을 TOOL_OK / MALFORMED / NO_TOOL 로 분류."""
    calls, source = extract_calls(resp)
    if calls:
        return "TOOL_OK", source, [n for n, _ in calls]
    text = to_text(resp.content)
    if ("<tool_call>" in text) or ('"arguments"' in text) or ('"parameters"' in text):
        return "MALFORMED", None, None               # 호출 의도는 있으나 파싱 실패
    return "NO_TOOL", None, None

def run_experiment(llm_with_tools, n=10):
    """같은 요청을 n 번 반복하고 분류 결과(및 호출된 도구 이름)를 반환한다."""
    results = []
    for i in range(n):
        try:
            resp = llm_with_tools.invoke(PROMPT)
            kind, source, names = classify(resp)
            detail = f"{names} ({source})" if names else to_text(resp.content)[:60].replace(chr(10), ' ')
        except Exception as e:
            kind, source, names, detail = "ERROR", None, None, repr(e)[:80]
        results.append({"kind": kind, "names": names or [], "detail": detail})
        print(f"  run {i+1:2d}: {kind:10s} | {detail}")
    return results

def summarize(results, label):
    """유효 tool_calls 비율 + '두 도구 모두 호출'(완성도) 비율을 집계."""
    c = Counter(r["kind"] for r in results)
    n = len(results)
    ok = c.get("TOOL_OK", 0)
    both = sum(1 for r in results if REQUIRED_TOOLS.issubset(set(r["names"])))  # 완성도 지표
    print(f"\n[{label}] 유효 tool_calls {ok}/{n} = {ok/n*100:.0f}%  ·  "
          f"두 도구 모두 호출 {both}/{n} = {both/n*100:.0f}%  ·  분류 {dict(c)}")
    return {"label": label, "n": n, "ok": ok, "both": both, "counts": dict(c)}

print("도구/도우미 준비 완료:", [t.name for t in TOOLS])

도구/도우미 준비 완료: ['get_weather', 'calculator']


## 3. 실험 ①-A — Ollama · 소형 (`qwen2.5:0.5b`)

Ollama 는 OpenAI 호환 `/v1` 에서 **소형 모델에도 도구 템플릿을 적용**해 **네이티브 `tool_calls`** 를
생성합니다 → 유효 호출률(TOOL_OK) ≈ 100%. (단, 다중 도구 요청의 **완성도**는 낮을 수 있음.)

바로 다음 **§3-2(①-B)** 에서 **완전히 같은 모델**을 `llama.cpp` 로 서빙해, 서버가 다르면
결과가 어떻게 달라지는지(런타임 축)를 대비합니다.

In [4]:
N = 10
llm_small = make_llm(MODELS["small"]["model"])
print(f"=== ① 로컬 {MODELS['small']['model']} 반복 실험 (N={N}) ===")
res_small = run_experiment(llm_small, N)
sum_small = summarize(res_small, MODELS["small"]["label"])

print("\n--- 참고: Ollama 는 소형 모델도 도구 템플릿을 적용해 '유효한' tool_calls 를 만든다 ---")
print("  (원시 llama.cpp 서버였다면 malformed <tool_call> 로 실패했을 가능성이 큼)")
incomplete = [x for x in res_small if not REQUIRED_TOOLS.issubset(set(x["names"]))]
if incomplete:
    print(f"  다만 요청의 두 도구 중 일부만 호출한(불완전) 경우가 {len(incomplete)}/{N} 건:")
    for r in incomplete[:3]:
        print("  •", r["detail"][:120])
else:
    print("  이번 실행에서는 매번 두 도구를 모두 호출했습니다(반복 시 편차 가능).")

=== ① 로컬 qwen2.5:0.5b 반복 실험 (N=10) ===


  run  1: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  2: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  3: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  4: TOOL_OK    | ['get_weather', 'calculator'] (native)


...

  run  6: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  7: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  8: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  9: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run 10: TOOL_OK    | ['get_weather', 'calculator'] (native)

[Ollama qwen2.5:0.5b] 유효 tool_calls 10/10 = 100%  ·  두 도구 모두 호출 9/10 = 90%  ·  분류 {'TOOL_OK': 10}

--- 참고: Ollama 는 소형 모델도 도구 템플릿을 적용해 '유효한' tool_calls 를 만든다 ---
  (원시 llama.cpp 서버였다면 malformed <tool_call> 로 실패했을 가능성이 큼)
  다만 요청의 두 도구 중 일부만 호출한(불완전) 경우가 1/10 건:
  • ['get_weather'] (native)


### 3-2. 실험 ①-B — **같은** 소형 모델을 `llama.cpp` 서버로 (런타임 축)

앞의 ①-A 는 **Ollama** 로 `qwen2.5:0.5b` 를 서빙했습니다. 이번엔 **완전히 같은 모델**을 보조 로컬
서버인 **`llama-cpp-python`** 으로 서빙합니다.

llama.cpp 기본 서버는 Ollama 와 달리 도구 호출을 **구조화(native `tool_calls`)로 파싱/포맷하지
않습니다.** 그래서 소형 모델이 도구 호출을 본문에 `<tool_call>{{ ... }}` 처럼 **깨진 JSON**(중괄호
중복 등)으로 흘리면, 파서가 실행 가능한 호출을 뽑지 못합니다 → **`MALFORMED`/`NO_TOOL`, 성공률 ≈ 0%.**

> 같은 모델인데 결과가 다른 이유 = **런타임(서버)**. GGUF 는 최초 1회만 받고(캐시), 서버는
> 백그라운드로 띄웠다가 실험 후 종료합니다(포트 8000, CPU).

In [5]:
# === 실험 ①-B: 같은 소형 모델(qwen2.5:0.5b)을 llama.cpp 서버로 서빙 → 도구 호출 실패 재현 ===
import time, requests
from huggingface_hub import hf_hub_download

# 1) 소형 모델 GGUF 다운로드(최초 1회, 이후 캐시)
GGUF = hf_hub_download(repo_id="Qwen/Qwen2.5-0.5B-Instruct-GGUF",
                       filename="qwen2.5-0.5b-instruct-q4_k_m.gguf")
print("GGUF:", GGUF)

# 2) llama.cpp OpenAI 호환 서버를 백그라운드로 기동 (:8000/v1, CPU)
LCPP = "http://127.0.0.1:8000/v1"
cmd = (f'"{sys.executable}" -m llama_cpp.server --model "{GGUF}" '
       f'--model_alias llamacpp-small --host 127.0.0.1 --port 8000 --n_ctx 4096 --n_gpu_layers 0')
utils.run_cmd_bg(cmd, "llamacpp_fc")

# 3) 서버 준비 대기(최대 ~120초)
ready = False
for _ in range(60):
    time.sleep(2)
    try:
        if requests.get(f"{LCPP}/models", timeout=2).ok:
            ready = True
            break
    except Exception:
        pass
print("llama.cpp 서버 준비:", ready)

if not ready:
    utils.tail_logs("llamacpp_fc", 20)
    sum_llamacpp = {"label": "llama.cpp qwen2.5:0.5b", "n": 0, "ok": 0, "both": 0, "counts": {}}
else:
    # 4) 같은 요청을 llama.cpp 서버의 소형 모델로 N회 반복 (Ollama 와 동일 인터페이스)
    llm_lcpp = ChatOpenAI(model="llamacpp-small", base_url=LCPP,
                          api_key="sk-no-key", temperature=0.7).bind_tools(TOOLS)
    print(f"=== ①-B llama.cpp qwen2.5:0.5b 반복 실험 (N={N}) ===")
    res_lcpp = run_experiment(llm_lcpp, N)
    sum_llamacpp = summarize(res_lcpp, "llama.cpp qwen2.5:0.5b")
    mal = [x for x in res_lcpp if x["kind"] in ("MALFORMED", "NO_TOOL")]
    if mal:
        print("\n--- 실행 불가(파싱 실패/미시도) 응답 예시 ---")
        for r in mal[:3]:
            print("  •", r["detail"][:140])

# 5) 서버 종료(자식 프로세스 포함) — 포트/자원 해제
utils.stop_bg("llamacpp_fc")

GGUF: C:\Users\stshin\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct-GGUF\snapshots\9217f5db79a29953eb74d5343926648285ec7e67\qwen2.5-0.5b-instruct-q4_k_m.gguf
[백그라운드 시작] name='llamacpp_fc'
$ "C:\Users\stshin\Documents\GitHub\Agentic AI Tutorial\.venv\Scripts\python.exe" -m llama_cpp.server --model "C:\Users\stshin\.cache\huggingface\hub\models--Qwen--Qwen2.5-0.5B-Instruct-GGUF\snapshots\9217f5db79a29953eb74d5343926648285ec7e67\qwen2.5-0.5b-instruct-q4_k_m.gguf" --model_alias llamacpp-small --host 127.0.0.1 --port 8000 --n_ctx 4096 --n_gpu_layers 0
  → 진행 로그는 다른 셀에서  utils.tail_logs('llamacpp_fc')  로 확인하세요.


llama.cpp 서버 준비: True
=== ①-B llama.cpp qwen2.5:0.5b 반복 실험 (N=10) ===


  run  1: MALFORMED  | <tool_call> {{"name": "get_weather", "arguments": {"city": "


  run  2: MALFORMED  | <tool_call> {{"name": "get_weather", "arguments": {"city": "


  run  3: MALFORMED  | <tool_call> {{"name": "get_weather", "arguments": {"city": "


...

  run  6: MALFORMED  | <tool_call> {{"name": "get_weather", "arguments": {"city": "
  run  7: MALFORMED  | <tool_call> {{"name": "get_weather", "arguments": {"city": "


  run  8: MALFORMED  | <tool_call> {{"name": "get_weather", "arguments": {"city": "


  run  9: MALFORMED  | <tool_call> {{"name": "get_weather", "arguments": {"city": "


  run 10: MALFORMED  | <tool_call> {{"name": "get_weather", "arguments": {"city": "

[llama.cpp qwen2.5:0.5b] 유효 tool_calls 0/10 = 0%  ·  두 도구 모두 호출 0/10 = 0%  ·  분류 {'MALFORMED': 10}

--- 실행 불가(파싱 실패/미시도) 응답 예시 ---
  • <tool_call> {{"name": "get_weather", "arguments": {"city": "
  • <tool_call> {{"name": "get_weather", "arguments": {"city": "
  • <tool_call> {{"name": "get_weather", "arguments": {"city": "


[중지] name='llamacpp_fc' (자식 프로세스 포함)


## 4. 실험 ② — 로컬 대형 모델 (`qwen3:8b`) ★ 핵심

같은 요청을 **`qwen3:8b`** 로 반복합니다. 8B 는 매번 **규격에 맞는 네이티브 `tool_calls`** 를
생성하므로 두 도구(`get_weather`, `calculator`) 호출을 안정적으로 추출합니다 → **성공률 ≈ 100%.**

In [6]:
llm_large = make_llm(MODELS["large"]["model"])
print(f"=== ② 로컬 {MODELS['large']['model']} 반복 실험 (N={N}) ===")
res_large = run_experiment(llm_large, N)
sum_large = summarize(res_large, MODELS["large"]["label"])

=== ② 로컬 qwen3:8b 반복 실험 (N=10) ===


  run  1: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  2: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  3: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  4: TOOL_OK    | ['get_weather', 'calculator'] (native)


...

  run  6: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  7: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  8: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  9: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run 10: TOOL_OK    | ['get_weather', 'calculator'] (native)

[Ollama qwen3:8b] 유효 tool_calls 10/10 = 100%  ·  두 도구 모두 호출 10/10 = 100%  ·  분류 {'TOOL_OK': 10}


### 4-1. 실제 도구 실행 루프 (`qwen3:8b`)

추출한 호출이 **진짜 실행 가능**함을 보입니다: 응답 파싱 → 파이썬 도구 실행 → 결과를 모델에 되먹여 최종 답변 생성.

In [7]:
resp = llm_large.invoke(PROMPT)
calls, source = extract_calls(resp)
print(f"1) 추출된 도구 호출 (source={source}):")
for name, args in calls:
    print(f"   - {name}({args})")

print("\n2) 도구 실제 실행:")
tool_results = []
for name, args in calls:
    out = TOOL_FUNCS[name].invoke(args)
    tool_results.append(f"{name} -> {out}")
    print(f"   - {out}")

print("\n3) 결과를 모델에 되먹여 최종 답변 생성:")
followup = PROMPT + [
    HumanMessage(content="도구 실행 결과입니다:\n" + "\n".join(tool_results) +
                 "\n이 결과만으로 사용자에게 한국어로 자연스럽게 최종 답변하세요. /no_think")]
# 최종 답변은 도구 없이 텍스트만 생성(기본 ollama/qwen3:8b)
final = utils.get_llm("ollama", temperature=0.3).invoke(followup)
print("   ", to_text(final.content).strip()[:300])

1) 추출된 도구 호출 (source=native):
   - get_weather({'city': '서울'})
   - calculator({'expression': 'sqrt(144)'})

2) 도구 실제 실행:
   - 서울 날씨: 맑음, 22도
   - sqrt(144) = 12.0

3) 결과를 모델에 되먹여 최종 답변 생성:


    서울의 날씨는 맑고, 현재 기온은 22도입니다. 또한, 144의 제곱근은 12.0입니다. 도움이 되었나요? 😊


## 5. 실험 ③ — 클라우드 (`google`)

동일한 요청을 클라우드 LLM 에 **N 번** 던집니다. 클라우드는 표준 `tool_calls`(native)를 안정적으로 반환합니다.

In [8]:
def pick_cloud():
    '''키가 설정된 클라우드 공급자를 고른다 (google→openai→anthropic).'''
    for p, key in [("google", utils.GOOGLE_API_KEY), ("openai", utils.OPENAI_API_KEY),
                   ("anthropic", utils.ANTHROPIC_API_KEY), ("anthropic_oauth", utils.ANTHROPIC_OAUTH_TOKEN)]:
        if key:
            return p
    return None

cloud = pick_cloud()
if cloud is None:
    print("클라우드 키가 없어 비교를 건너뜁니다. .env 에 GOOGLE_API_KEY 등을 설정하세요.")
    res_cloud, sum_cloud = [], {"label": "클라우드(없음)", "n": 0, "ok": 0, "counts": {}}
else:
    llm_cloud = utils.get_llm(cloud, temperature=0.7).bind_tools(TOOLS)
    print(f"=== 클라우드({cloud}) 반복 실험 (N={N}) ===")
    res_cloud = run_experiment(llm_cloud, N)
    sum_cloud = summarize(res_cloud, f"클라우드:{cloud}")

=== 클라우드(google) 반복 실험 (N=10) ===


  run  1: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  2: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  3: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  4: TOOL_OK    | ['get_weather', 'calculator'] (native)


...

  run  6: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  7: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  8: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run  9: TOOL_OK    | ['get_weather', 'calculator'] (native)


  run 10: TOOL_OK    | ['get_weather', 'calculator'] (native)

[클라우드:google] 유효 tool_calls 10/10 = 100%  ·  두 도구 모두 호출 10/10 = 100%  ·  분류 {'TOOL_OK': 10}


## 6. 비교 결과 (3자)

In [9]:
def bar(pct, width=20):
    fill = int(round(pct / 100 * width))
    return "█" * fill + "·" * (width - fill)

# 런타임 축(①-A vs ①-B) + 모델 축(①-A vs ②)을 한눈에
rows = [sum_small, sum_llamacpp, sum_large, sum_cloud]
print(f"{'런타임 · 모델':<24}{'유효 tool_calls':<26}{'두 도구 모두(완성도)'}")
print("-" * 88)
for s in rows:
    if not s or s.get("n", 0) == 0:
        continue
    ok_pct = s["ok"] / s["n"] * 100
    both_pct = s["both"] / s["n"] * 100
    print(f"{s['label']:<24}{bar(ok_pct)} {ok_pct:3.0f}%   {bar(both_pct)} {both_pct:3.0f}%")

print("\n해석 — function calling 신뢰도의 '두 축':")
print("  (런타임 축) Ollama qwen2.5:0.5b  vs  llama.cpp qwen2.5:0.5b  ← 같은 모델, 다른 서버")
print("     · Ollama: 도구 템플릿 적용 → 유효 tool_calls / llama.cpp 기본: 미파싱 → malformed·실패")
print("  (모델 축)   qwen2.5:0.5b  vs  qwen3:8b  ← 같은 서버(Ollama), 다른 모델")
print("     · 큰 모델일수록 다중 도구 태스크의 완성도·인자 정확도가 높다")
print("  ⇒ '로컬이라 안 된다'가 아니라, 적절한 런타임(Ollama) + 충분한 모델(qwen3:8b)이면 로컬도 안정적.")


런타임 · 모델                유효 tool_calls             두 도구 모두(완성도)
----------------------------------------------------------------------------------------
Ollama qwen2.5:0.5b     ████████████████████ 100%   ██████████████████··  90%
llama.cpp qwen2.5:0.5b  ····················   0%   ····················   0%
Ollama qwen3:8b         ████████████████████ 100%   ████████████████████ 100%
클라우드:google             ████████████████████ 100%   ████████████████████ 100%

해석 — function calling 신뢰도의 '두 축':
  (런타임 축) Ollama qwen2.5:0.5b  vs  llama.cpp qwen2.5:0.5b  ← 같은 모델, 다른 서버
     · Ollama: 도구 템플릿 적용 → 유효 tool_calls / llama.cpp 기본: 미파싱 → malformed·실패
  (모델 축)   qwen2.5:0.5b  vs  qwen3:8b  ← 같은 서버(Ollama), 다른 모델
     · 큰 모델일수록 다중 도구 태스크의 완성도·인자 정확도가 높다
  ⇒ '로컬이라 안 된다'가 아니라, 적절한 런타임(Ollama) + 충분한 모델(qwen3:8b)이면 로컬도 안정적.


## 7. 멀티(병렬) Function Calling — 에이전트를 위한 핵심 능력

앞 실험(§1~6)은 *"유효한 도구 호출을 만드느냐"*(신뢰도)를 봤습니다. 여기서는 한 걸음 더 나아가
**"한 번의 응답에서 여러 도구를 동시에 호출할 수 있느냐"** 를 봅니다.

- **단일(single) function calling**: 한 응답에 도구를 **하나만** 요청 → 나머지는 다음 턴에.
- **멀티(병렬, parallel) function calling**: 서로 **독립적인** 여러 도구를 **한 응답에서 동시에** 요청.

### 왜 에이전트에 중요한가
독립된 하위 작업(예: "날씨 조회" + "제곱근 계산")은 서로 순서 의존이 없습니다. 멀티 호출은 이를 **한 턴에**
모아 요청하므로:
- **왕복(round-trip)·지연 감소** — LLM 호출 횟수가 줄어 더 빠르고 저렴하다.
- **동시 계획** — 모델이 필요한 행동들을 한꺼번에 세워 도구 오케스트레이션이 단순해진다.
- **ReAct/도구 에이전트 효율** — 여러 관찰을 한 번에 모아 다음 추론으로 넘어간다.

단일만 되는 모델도 에이전트 루프를 여러 번 돌면 결국 두 작업을 처리하지만, 턴 수가 늘어 느리고 실패
지점이 많습니다. 그래서 **병렬 호출 능력은 실용적인 에이전트 설계에서 중요한 요소**입니다.

### 실측 요지 (아래 셀에서 직접 반복 측정)
- **NVIDIA `meta/llama-3.1-8b-instruct` · Google Gemini · Ollama `qwen3:8b`** 는 이 요청에서 **대체로
  한 응답에 두 도구를 병렬로** 호출했다 ✅. *"NVIDIA(또는 특정 모델)는 단일만 된다"* 는 통념과 달리,
  이 실험에서 **단일 전용 공급자는 나타나지 않았다.**
- 다만 호출 개수는 **런·프롬프트·온도에 따라 편차**가 있어 가끔 1개(단일)로 떨어질 수 있다(반복 실행으로 확인).

> 📌 멀티/단일 여부는 *모델·런타임·프롬프트·온도* 가 함께 좌우하며 **비결정적 편차**가 있습니다. 따라서
> 특정 모델을 에이전트에 쓰기 전, **아래처럼 직접 N회 측정**해 병렬 호출이 안정적인지 확인하는 습관이 중요합니다.
> (앞 §1~6 은 출력 정돈을 위해 `/no_think` 를 썼지만, 병렬 호출 개수에는 사고 ON/OFF 가 큰 차이를 주지 않았습니다.)

In [10]:
# === §7. 멀티(병렬) function calling 측정 — 한 응답에서 도구를 몇 개 호출하는가 ===
from langchain_core.messages import HumanMessage, SystemMessage

# 서로 독립적인 두 도구(get_weather·calculator)가 모두 필요한 요청.
# 같은 qwen3:8b 를 사고 ON/OFF(/no_think)로도 넣어 사고 모드의 영향(편차)까지 관찰한다.
MULTI_USER = "서울 날씨를 알려주고 144의 제곱근을 계산해줘."
P_THINK   = [SystemMessage(content="반드시 제공된 도구를 사용하세요."),           HumanMessage(content=MULTI_USER)]
P_NOTHINK = [SystemMessage(content="반드시 제공된 도구를 사용하세요. /no_think"), HumanMessage(content=MULTI_USER)]

def count_parallel(llm_with_tools, prompt, n=8):
    """같은 요청을 n회 던져 '단일 응답에서 반환된 tool_calls 개수'를 기록한다."""
    counts = []
    for _ in range(n):
        try:
            resp = llm_with_tools.invoke(prompt)
            counts.append(len(getattr(resp, "tool_calls", None) or []))
        except Exception as e:
            counts.append(f"ERR:{type(e).__name__}")
    return counts

def verdict(counts):
    """응답별 호출 개수로 판정: 하나라도 2+면 병렬 가능, 전부 1이면 단일, 편차는 별도 표시."""
    nums = [c for c in counts if isinstance(c, int)]
    if not nums:
        return "도구 미호출 ❓"
    multi = sum(1 for c in nums if c >= 2)
    if multi == 0:
        return "단일만 ⚠️"
    tag = "멀티(병렬) ✅" if multi == len(nums) else f"주로 멀티(편차 있음: {multi}/{len(nums)}) ✅"
    return tag

# 측정 대상: 같은 qwen3:8b 를 사고 ON/OFF 로, 그리고 클라우드 두 곳(키 있으면)
Nm = 8
trials = [
    ("Ollama qwen3:8b (사고 ON)",   make_llm("qwen3:8b", 0), P_THINK),
    ("Ollama qwen3:8b (/no_think)", make_llm("qwen3:8b", 0), P_NOTHINK),
]
if utils.NVIDIA_API_KEY:
    trials.append(("NVIDIA " + utils.NVIDIA_MODEL, utils.get_llm("nvidia", 0).bind_tools(TOOLS), P_THINK))
else:
    print("NVIDIA_API_KEY 미설정 — build.nvidia.com 키를 .env 에 넣으면 함께 비교됩니다.")
if utils.GOOGLE_API_KEY:
    trials.append(("Google Gemini", utils.get_llm("google", 0).bind_tools(TOOLS), P_THINK))
else:
    print("GOOGLE_API_KEY 미설정 — 클라우드 비교를 건너뜁니다.")

print(f"요청: {MULTI_USER}\n(독립된 두 도구 get_weather·calculator 가 모두 필요, 각 {Nm}회 반복)\n")
for label, llm, prompt in trials:
    counts = count_parallel(llm, prompt, Nm)
    print(f"[{label:30s}] 응답별 tool_calls {counts} → {verdict(counts)}")

print("\n해석:")
print("  · NVIDIA llama-3.1-8b · Google · Ollama qwen3:8b 모두 대체로 한 응답에 두 도구를 병렬로 호출 → 멀티")
print("  · qwen3:8b 는 사고 ON/OFF(/no_think) 모두 대체로 멀티 — 병렬 호출에 사고 모드 차이는 크지 않았다")
print("  · 호출 개수는 모델·프롬프트·온도에 따라 런마다 달라져 가끔 1개(단일)가 섞일 수 있다(비결정적 편차)")
print("  ⇒ '특정 모델은 단일만 된다'고 단정 말고 쓸 모델로 직접 측정하라. 병렬 호출은 왕복·지연을 줄여 에이전트에 중요.")

요청: 서울 날씨를 알려주고 144의 제곱근을 계산해줘.
(독립된 두 도구 get_weather·calculator 가 모두 필요, 각 8회 반복)



[Ollama qwen3:8b (사고 ON)       ] 응답별 tool_calls [2, 2, 2, 2, 2, 2, 2, 2] → 멀티(병렬) ✅


[Ollama qwen3:8b (/no_think)   ] 응답별 tool_calls [2, 2, 2, 2, 2, 2, 2, 2] → 멀티(병렬) ✅


[NVIDIA meta/llama-3.1-8b-instruct] 응답별 tool_calls [2, 2, 2, 2, 2, 2, 2, 2] → 멀티(병렬) ✅


[Google Gemini                 ] 응답별 tool_calls [2, 2, 2, 2, 2, 2, 2, 2] → 멀티(병렬) ✅

해석:
  · NVIDIA llama-3.1-8b · Google · Ollama qwen3:8b 모두 대체로 한 응답에 두 도구를 병렬로 호출 → 멀티
  · qwen3:8b 는 사고 ON/OFF(/no_think) 모두 대체로 멀티 — 병렬 호출에 사고 모드 차이는 크지 않았다
  · 호출 개수는 모델·프롬프트·온도에 따라 런마다 달라져 가끔 1개(단일)가 섞일 수 있다(비결정적 편차)
  ⇒ '특정 모델은 단일만 된다'고 단정 말고 쓸 모델로 직접 측정하라. 병렬 호출은 왕복·지연을 줄여 에이전트에 중요.


## 8. 정리

function calling 신뢰도는 **두 축**에 함께 달려 있고(§1~6), 여기에 에이전트 관점의 **세 번째 축**(§7)을 더했습니다.

1. **런타임/서버 축** — 실험 ①-A vs ①-B (**같은** `qwen2.5:0.5b`, 다른 서버):
   - **Ollama**: OpenAI 호환 `/v1` 에서 도구 템플릿을 적용해 *유효한* 네이티브 `tool_calls` 생성 ✅
   - **llama.cpp**(기본 서버): 도구 호출을 구조화 파싱하지 않아 소형 모델이 `<tool_call>{{...}}` 같은
     **깨진 JSON** 을 흘려 실행 불가 ❌  → *같은 모델도 서버에 따라 결과가 다르다*.
2. **모델 능력 축** — 실험 ①-A vs ② (같은 Ollama, 다른 모델):
   - 큰 모델(`qwen3:8b`)일수록 다중 도구 태스크의 **완성도·인자 정확도**가 높다.
3. **멀티(병렬) 호출 축** — 실험 §7 (한 응답에서 도구를 몇 개 호출하나):
   - **NVIDIA `llama-3.1-8b` · Google Gemini · Ollama `qwen3:8b`** 모두 이 요청에서 대체로 **두 도구를
     병렬로** 호출했다 ✅. *"특정 모델은 단일만"* 이라는 통념과 달리 **단일 전용 공급자는 없었고**,
     호출 개수는 런·프롬프트·온도에 따라 **비결정적 편차**가 있다(가끔 1개).
   - 병렬 호출은 왕복·지연을 줄이고 동시 계획을 가능케 해 **실용적 에이전트 설계에서 중요**하다.
     따라서 특정 모델을 쓰기 전 **직접 반복 측정**해 병렬 호출 안정성을 확인하는 습관이 좋다.

**결론**: *"로컬이라 안 된다"* 가 아니라, **적절한 런타임(Ollama) + 충분한 모델(`qwen3:8b`)** 이면
로컬로도 안정적인 function calling 이 가능하다. (llama.cpp 를 쓰더라도 네이티브 도구 파싱을 지원하는
`llama-server --jinja` 나 `--chat_format` 을 맞추면 개선된다.) 나아가 **멀티(병렬) 호출** 능력까지 보면,
NVIDIA build·Google·로컬 `qwen3:8b` 모두 에이전트에 필요한 병렬 도구 호출을 (대체로) 지원한다. 메인
노트북(`M02_3_mcp_a2a.ipynb` §7)은 기본 공급자를 그대로 쓰는 `utils.get_llm()` 으로 에이전트를 돌리므로,
`qwen3:8b` 를 로컬에 두면 도구 호출까지 로컬(Ollama)로 처리할 수 있다.